In [1]:
import sys, pathlib, warnings
warnings.filterwarnings("ignore")
_r = pathlib.Path.cwd().resolve()
while not (_r / "config" / "settings.yaml").exists() and _r != _r.parent:
    _r = _r.parent
sys.path.insert(0, str(_r))                      # makes `import src...` work from any folder
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import plotly.express as px
from src.notebook_utils import *
sns.set_theme(style="whitegrid"); pd.set_option("display.max_columns", 40, "display.width", 140)
print("project root:", _r)

project root: /app


# 03 - Supervised fraud detection
Logistic Regression, Random Forest and XGBoost on a **chronological** split (train 70% / validation 15% / test 15%).

In [2]:
from src.fraud_detection.train import train_and_select, evaluate_models
from src.model_evaluation.evaluate import comparison_table
import copy
fe = load_features(); d = split_xy(fe)
X_tr, y_tr, _ = d['train']; X_va, y_va, _ = d['validation']; X_te, y_te, te = d['test']
print({k: (len(v[1]), int(v[1].sum())) for k, v in d.items()}, '(rows, frauds)')

{'train': (41777, 835), 'validation': (8952, 197), 'test': (8953, 174)} (rows, frauds)


In [3]:
cfg = copy.deepcopy(CFG); cfg['tuning']['n_iter'] = 3   # quick demo; the pipeline uses the value in settings.yaml
trained = train_and_select(X_tr, y_tr, X_va, y_va, cfg)
results, scores = evaluate_models(trained, X_te, y_te)
comparison_table(results).round(4)

2026-11-09 06:26:20,215 INFO    src.fraud_detection.train | trained logistic_regression in 9.5s (cv average_precision=0.6178, thr=0.941)


2026-11-09 06:27:03,104 INFO    src.fraud_detection.train | trained random_forest in 42.8s (cv average_precision=0.7226, thr=0.671)


2026-11-09 06:27:11,963 INFO    src.fraud_detection.train | trained xgboost in 8.8s (cv average_precision=0.7251, thr=0.902)


,accuracy,precision,recall,f1,roc_auc,pr_auc,false_positive_rate,tp,fp,fn,tn
xgboost,0.9925,0.982,0.6264,0.7649,0.8854,0.7429,0.0002,109.0,2.0,65.0,8777.0
random_forest,0.9925,0.928,0.6667,0.7759,0.9026,0.7412,0.0010,116.0,9.0,58.0,8770.0
logistic_regression,0.9899,0.875,0.5632,0.6853,0.8729,0.6669,0.0016,98.0,14.0,76.0,8765.0


In [4]:
pd.DataFrame({k: v['params'] for k, v in trained.items()}).T

,clf__C,max_depth,max_features,min_samples_leaf,colsample_bytree,learning_rate,min_child_weight,subsample
logistic_regression,0.074593,NaN,NaN,NaN,NaN,NaN,NaN,NaN
random_forest,NaN,10,sqrt,16,NaN,NaN,NaN,NaN
xgboost,NaN,5.0,NaN,NaN,0.662407,0.030514,8.0,0.733483


In [5]:
from sklearn.metrics import PrecisionRecallDisplay, RocCurveDisplay
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for k, s in scores.items():
    RocCurveDisplay.from_predictions(y_te, s, name=k, ax=ax[0]); PrecisionRecallDisplay.from_predictions(y_te, s, name=k, ax=ax[1])
ax[0].set_title('ROC'); ax[1].set_title('Precision-Recall')

Text(0.5, 1.0, 'Precision-Recall')

In [6]:
print('Pipeline (full-tuning) results for comparison:')
comparison_table({k: v for k, v in load_summary()['models']['results'].items() if k in trained}).round(4)

Pipeline (full-tuning) results for comparison:


,accuracy,precision,recall,f1,roc_auc,pr_auc,false_positive_rate,tp,fp,fn,tn
xgboost,0.9925,0.9820,0.6264,0.7649,0.8854,0.7429,0.0002,109.0,2.0,65.0,8777.0
random_forest,0.9925,0.9280,0.6667,0.7759,0.9026,0.7412,0.0010,116.0,9.0,58.0,8770.0
logistic_regression,0.9901,0.8829,0.5632,0.6877,0.8722,0.6658,0.0015,98.0,13.0,76.0,8766.0


Class imbalance (~2%) is handled by `class_weight` / `scale_pos_weight`; thresholds are tuned on the **validation** set so the test set remains untouched.